# Distributed Compute with Dask / Ray

`pip install "dask[complete]" ray pandas`. Sections degrade gracefully if one of the
two frameworks isn't installed.

## 1. A pandas baseline to beat

In [ ]:
import time

import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
N = 5_000_000
pdf = pd.DataFrame({
    "group": rng.integers(0, 1000, N),
    "x": rng.normal(size=N),
    "y": rng.uniform(size=N),
})

t0 = time.perf_counter()
pandas_result = pdf.groupby("group").agg(x_mean=("x", "mean"), y_sum=("y", "sum"))
t_pandas = time.perf_counter() - t0
print(f"pandas groupby: {t_pandas:.2f}s")

## 2. Dask DataFrame: same API, chunked and lazy

In [ ]:
import dask.dataframe as dd

ddf = dd.from_pandas(pdf, npartitions=8)
print("npartitions:", ddf.npartitions)

lazy = ddf.groupby("group").agg(x_mean=("x", "mean"), y_sum=("y", "sum"))
print("nothing computed yet — lazy object:", type(lazy).__name__)

t0 = time.perf_counter()
dask_result = lazy.compute()
t_dask = time.perf_counter() - t0
print(f"dask groupby:   {t_dask:.2f}s (vs pandas {t_pandas:.2f}s)")
# On one machine with data that fits in RAM, dask may not beat pandas —
# that's lesson one. Its value: data that does NOT fit, or a real cluster.

## 3. The dashboard: watch the task graph execute

In [ ]:
from dask.distributed import Client

client = Client(processes=False)  # in-process scheduler; prints dashboard link
print("dashboard:", client.dashboard_link)

# Re-run the computation and watch the task stream at the dashboard link:
_ = dd.from_pandas(pdf, npartitions=8).groupby("group").x.mean().compute()
client.close()

## 4. Ray: parallelize arbitrary functions, then hold state in an actor

In [ ]:
try:
    import ray
    ray.init(ignore_reinit_error=True, num_cpus=4, include_dashboard=False)

    @ray.remote
    def slow_square(x):
        time.sleep(0.5)
        return x * x

    t0 = time.perf_counter()
    futures = [slow_square.remote(i) for i in range(8)]  # returns immediately
    print("futures created in", round(time.perf_counter() - t0, 3), "s")
    print("results:", ray.get(futures), f"in {time.perf_counter() - t0:.2f}s (8 x 0.5s of work)")

    @ray.remote
    class Counter:
        def __init__(self):
            self.total = 0
        def add(self, value):
            self.total += value
            return self.total

    counter = Counter.remote()
    ray.get([counter.add.remote(i) for i in range(10)])
    print("actor state after 10 serialized calls:", ray.get(counter.add.remote(0)))
    ray.shutdown()
except ImportError:
    print("ray not installed — pip install ray")

## 5. Mini-project: the three-way comparison

In [ ]:
summary = pd.DataFrame({
    "wall_time_s": {"pandas": round(t_pandas, 2), "dask (8 partitions)": round(t_dask, 2)},
})
summary
# Extend this: (1) grow N until pandas hits memory pressure and dask keeps working
# (use dd.from_map or read from partitioned parquet to avoid materializing in pandas);
# (2) implement the same groupby with ray tasks (one task per group-chunk) and add it;
# (3) note code complexity alongside the timings — that's part of the tradeoff.